# 06. Final Query Construction, Balancing, and Audit — Herbal Supplements

This notebook defines the final synthetic-query pipeline for the full eligible reserve created by Notebook 03. Positive query evidence is limited to query-safe signals extracted from the held-out target review by Notebook 05. Deterministic seeds are assembled in a fixed family order from ingredient or herb, benefit or need, form, and claim or dietary-constraint signals. Flavor is retained for audit but excluded from the active query seed.

Target-item title, Brand, seller or manufacturer terms, identifiers, and package or dosage information are joined only after seed construction. They serve exclusively as a negative dictionary for cue removal and leakage audit; they cannot add, repair, or supplement query content. Catalog metadata, prior user history, frozen population review-derived item signals, ratings, and sentiment are not positive query evidence.

Before any language-model call, the notebook applies the query-sufficiency, length, language-safety, metadata-safety, and downstream prior-item requirements. It fixes the balanced quota from the smallest downstream-eligible regime and uses Notebook 03’s deterministic within-regime order for same-regime replacement. The thesis and executed downstream artifacts identify the canonical benchmark as 1,968 cases: 656 cold, 656 weak, and 656 strong.

The DSPy step is a seed-only linguistic rewrite. An output is accepted only if it remains within the permitted length, adds no new content concepts, preserves protected multiword phrases, and passes every direct-cue check. Otherwise, the deterministic review-safe seed is retained. The active query is stored in `query`; `query_C` is an exact compatibility alias, `query_seed` preserves the deterministic source, and `query_clean` remains inactive.

The received notebook contains 11 unexecuted code cells and no stored outputs. Its default configuration performs an identity-only preflight and blocks canonical publication. It therefore documents the implementation and publication gate but does not by itself evidence the canonical query-generation run.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip install -q pyarrow dspy

In [ ]:
# ==== Load Libraries ====
from pathlib import Path
import html
import json
import re
import time

import dspy
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 240)

In [ ]:
# ==== Define Inputs, Query Policy, and Preflight Controls ====
PROJECT_ROOT = Path("/content/drive/MyDrive/thesis_recsys/categories/herbal_supplements")

ELIGIBLE_POOL_PATH = PROJECT_ROOT / "data/interim/user_regime_sampling/herbal_sampled_query_cases.parquet"
TRAIN_PRIOR_HISTORY_PATH = PROJECT_ROOT / "data/processed/user_sampling/herbal_user_prior_review_history_training.parquet"
REVIEW_SIGNAL_PATH = PROJECT_ROOT / "data/processed/review_signals/herbal_review_signals.parquet"
ITEM_SCHEMA_PATH = PROJECT_ROOT / "data/processed/items/herbal_item_schema.parquet"
ITEM_DOCS_PATH = PROJECT_ROOT / "data/processed/items/item_docs_herbal.parquet"

OUTPUT_DIR = PROJECT_ROOT / "outputs/query_cache"
QUERY_CACHE_PATH = OUTPUT_DIR / "herbal_query_cache.parquet"
QUERY_CACHE_CSV_PATH = OUTPUT_DIR / "herbal_query_cache.csv"
QUERY_CONTRACT_PATH = OUTPUT_DIR / "herbal_query_generation_config.json"
QUERY_SUMMARY_PATH = OUTPUT_DIR / "herbal_query_generation_summary_medium_heavy_dspy.json"
QUERY_QC_PATH = OUTPUT_DIR / "herbal_query_generation_qc_medium_heavy_dspy.parquet"
PRE_GENERATION_FAILURE_REASON_PATH = OUTPUT_DIR / "herbal_query_pre_generation_failure_reason_summary.csv"
PRE_GENERATION_ATTRITION_QC_PATH = OUTPUT_DIR / "herbal_query_pre_generation_attrition_qc.csv"

REGIME_ORDER = ["cold", "weak", "strong"]
EXPECTED_ELIGIBLE_REGIME_COUNTS = None
EXPECTED_ELIGIBLE_ROWS = None
EXPECTED_TARGET_PER_REGIME = None
EXPECTED_OUTPUT_ROWS = None
EXPECTED_TARGET_SELECTION_MODE = "recent_eligible_review_rank_le5"
MAX_TARGET_RANK_ALLOWED = 5

QUERY_VARIANT = "C_medium_heavy_clean_dspy_linguistic_coverage_preserving"
ELIGIBILITY_POLICY = "coverage_preserving_5t_1f_1s"
RANDOM_SEED = 42

MIN_QUERY_TOKENS = 5
MAX_QUERY_TOKENS = 18
MAX_RESIDUAL_TOKENS = 40
RESIDUAL_PAD_TARGET_TOKENS = MIN_QUERY_TOKENS
MIN_SPECIFIC_FAMILIES = 1
MIN_SPECIFIC_SIGNALS = 1

ACTIVE_ELIGIBILITY_POLICY = {
    "min_tokens": MIN_QUERY_TOKENS,
    "min_families": MIN_SPECIFIC_FAMILIES,
    "min_signals": MIN_SPECIFIC_SIGNALS,
}
TITLE_OVERLAP_WARNING_THRESHOLD = 0.45
RESIDUAL_TEXT_COLUMN = "query_safe_text"

RUN_SMOKE_TEST = False
SMOKE_TEST_N = 5
WRITE_OUTPUTS_IN_SMOKE_TEST = False

# Default to a non-publishing identity preflight.
# Canonical publication requires both an explicit publication setting
# and a successful locked-order identity check.
RUN_IDENTITY_ONLY_PREFLIGHT = True
ALLOW_CANONICAL_PUBLICATION = False
IDENTITY_PREFLIGHT_PASSED = False

USE_DSPY_LINGUISTIC_REWRITE = True
FAIL_IF_DSPY_UNAVAILABLE = True
DEEPSEEK_MODEL = "openai/deepseek-chat"
DEEPSEEK_BASE_URL = "https://api.deepseek.com/v1"
DEEPSEEK_COLAB_SECRET = "deepseek_api_key"
DSPY_TEMPERATURE = 0.0
DSPY_MAX_TOKENS = 80
DSPY_MAX_RETRIES = 2
DSPY_RETRY_SLEEP_SECONDS = 1.5
DSPY_MODE = "linguistic_rewrite_only"

FAMILY_SPECS = [
    ("ingredient_or_herb", "ingredient_or_herb_signals", 2),
    ("benefit_need", "benefit_need_signals", 2),
    ("form", "form_signals", 1),
    ("claim_diet", "claim_diet_signals", 2),
]
EXCLUDED_QUERY_FAMILIES = ["flavor"]

CATEGORY_ANCHOR_PHRASES = {
    "herbal supplement", "herbal supplements", "dietary supplement",
    "dietary supplements", "herbal", "supplement", "supplements",
}
CATEGORY_ANCHOR_TOKENS = {"herbal", "supplement", "supplements"}
GENERIC_UTILITY_TOKENS = {
    "support", "supports", "help", "helps", "promote", "promotes", "boost",
    "wellness", "natural", "formula", "blend", "complex", "routine", "care",
    "product", "products", "solution", "solutions",
}
CONTEXT_DEPENDENT_TOKENS = {"daily", "health"}
RATING_SENTIMENT_TOKENS = {
    "amazing", "awesome", "bad", "best", "better", "effective", "excellent",
    "favorite", "good", "great", "hate", "like", "love", "perfect",
    "recommend", "recommended", "terrible", "value", "wonderful", "works",
    "worked", "working", "rating", "rated", "star", "stars",
}
FUNCTION_WORDS = {
    "for", "with", "without", "and", "or", "to", "of", "in", "on", "as", "by",
    "is", "are", "that", "the", "a", "an",
}

METADATA_GENERIC_TOKENS = {
    "herbal", "supplement", "supplements", "support", "capsules", "capsule",
    "gummies", "gummy", "tea", "powder", "drops", "drop", "softgel",
    "softgels", "liquid", "extract", "organic", "vegan", "natural", "immune",
    "sleep", "stress", "digestive", "joint", "energy", "focus", "calm",
    "relaxation", "wellness", "root", "leaf", "berry", "store", "official",
}
METADATA_NAME_COLUMNS = [
    "itemctx_facet_brand_text", "itemctx_manufacturer", "itemctx_store",
]
METADATA_DIAGNOSTIC_COLUMNS = [
    "itemctx_identifier_diagnostic_text", "itemctx_package_dimensions",
    "itemctx_unit_count", "itemctx_number_of_items", "itemctx_item_model_number",
    "target_parent_asin",
]

TOKEN_EQUIVALENCE = {
    "supports": "support",
    "supporting": "support",
    "supported": "support",
    "support": "support",
    "supplements": "supplement",
    "supplement": "supplement",
    "herbs": "herbal",
    "herb": "herbal",
    "herbal": "herbal",
    "capsules": "capsule",
    "capsule": "capsule",
    "tablets": "tablet",
    "tablet": "tablet",
    "softgels": "softgel",
    "softgel": "softgel",
    "gummies": "gummy",
    "gummy": "gummy",
    "drops": "drop",
    "drop": "drop",
    "liquid": "liquid",
    "liquids": "liquid",
    "powders": "powder",
    "powder": "powder",
    "teas": "tea",
    "tea": "tea",
    "extracts": "extract",
    "extract": "extract",
    "tinctures": "tincture",
    "tincture": "tincture",
    "mushrooms": "mushroom",
    "mushroom": "mushroom",
    "roots": "root",
    "root": "root",
    "leaves": "leaf",
    "leaf": "leaf",
    "berries": "berry",
    "berry": "berry",
    "immune": "immune",
    "immunity": "immune",
    "digestive": "digestion",
    "digestion": "digestion",
    "joints": "joint",
    "joint": "joint",
    "relax": "relaxation",
    "relaxing": "relaxation",
    "relaxation": "relaxation",
    "sleep": "sleep",
    "energy": "energy",
    "stress": "stress",
    "calm": "calm",
    "focus": "focus",
}

ASIN_PATTERN = re.compile(r"\bB0[A-Z0-9]{8}\b|\bB[0-9A-Z]{9}\b", re.IGNORECASE)
PACKAGE_PATTERN = re.compile(
    r"\b(\d+(?:\.\d+)?\s?(?:mg|mcg|iu|g|gram|grams|ml|oz|fl\.?\s?oz|ct|count|capsule|capsules|tablet|tablets|softgel|softgels|gummy|gummies|serving|servings|pack|packs|bottle|bottles)|asin|sku|upc|barcode|seller|manufacturer)\b",
    re.IGNORECASE,
)
SELLER_PATTERN = re.compile(
    r"\b(sold by|seller|manufacturer|made by|distributed by|shipped by)\b",
    re.IGNORECASE,
)

for path in [ELIGIBLE_POOL_PATH, TRAIN_PRIOR_HISTORY_PATH, REVIEW_SIGNAL_PATH, ITEM_SCHEMA_PATH, ITEM_DOCS_PATH]:
    if not path.exists():
        raise FileNotFoundError(path)

print("Input:", ELIGIBLE_POOL_PATH)
print("Input:", TRAIN_PRIOR_HISTORY_PATH)
print("Input:", REVIEW_SIGNAL_PATH)
print("Input:", ITEM_SCHEMA_PATH)
print("Input:", ITEM_DOCS_PATH)
print("Output:", QUERY_CACHE_PATH)


In [ ]:
# ==== Define Seed, Cue-Removal, and Validation Helpers ====
def normalize_space(value):
    if value is None or pd.isna(value):
        return ""
    text = html.unescape(str(value))
    text = re.sub(r"<br\s*/?>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    return re.sub(r"\s+", " ", text.replace("\n", " ").replace("\t", " ")).strip()


def to_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if value is None or pd.isna(value):
        return False
    return str(value).strip().lower() in {"true", "1", "yes"}


def to_int(value, default=0):
    numeric = pd.to_numeric(value, errors="coerce")
    return default if pd.isna(numeric) else int(numeric)


def tokenize(value):
    return re.findall(r"[a-z0-9']+", normalize_space(value).lower())


def token_count(value):
    return len(tokenize(value))


def split_pipe_values(value):
    text = normalize_space(value)
    if not text:
        return []
    values = []
    seen = set()
    for part in re.split(r"\s*\|\s*", text):
        cleaned = normalize_space(part).lower()
        if cleaned and cleaned not in seen:
            values.append(cleaned)
            seen.add(cleaned)
    return values


def pipe_join(values):
    out = []
    seen = set()
    for value in values:
        cleaned = normalize_space(value).lower()
        if cleaned and cleaned not in seen:
            out.append(cleaned)
            seen.add(cleaned)
    return " | ".join(out)


def normalize_content_token(token):
    return TOKEN_EQUIVALENCE.get(token, token)


def content_tokens(value):
    return {
        normalize_content_token(token)
        for token in tokenize(value)
        if token not in FUNCTION_WORDS
    }


def phrase_supported(query, phrase):
    phrase_concepts = content_tokens(phrase)
    return bool(phrase_concepts) and phrase_concepts.issubset(content_tokens(query))


def normalize_query_text(value):
    text = normalize_space(value).lower().replace("&", " and ")
    text = re.sub(r"[^a-z0-9\s\-']", " ", text)
    return re.sub(r"\s+", " ", text).strip(" -")


def split_metadata_values(value):
    text = normalize_space(value)
    if not text:
        return []
    return [
        normalize_space(part).lower()
        for part in re.split(r"\s*\|\s*|\s*;\s*|\s*,\s*", text)
        if normalize_space(part)
    ]


def remove_exact_phrase(text, phrase):
    phrase_tokens = tokenize(phrase)
    if not phrase_tokens:
        return normalize_query_text(text)
    pattern = r"(?<![a-z0-9])" + r"[\s_\-–—]+".join(
        re.escape(token) for token in phrase_tokens
    ) + r"(?:['’]s)?(?![a-z0-9])"
    return normalize_query_text(re.sub(pattern, " ", normalize_query_text(text), flags=re.IGNORECASE))


def exact_phrase_present(text, phrase):
    normalized = normalize_query_text(text)
    return bool(normalized) and remove_exact_phrase(normalized, phrase) != normalized


def is_specific_phrase(phrase):
    normalized = normalize_query_text(phrase)
    tokens = tokenize(normalized)
    if not tokens:
        return False
    if any(token in RATING_SENTIMENT_TOKENS for token in tokens):
        return False
    if normalized in CATEGORY_ANCHOR_PHRASES:
        return False
    if len(tokens) == 1 and tokens[0] in (
        CATEGORY_ANCHOR_TOKENS
        | GENERIC_UTILITY_TOKENS
        | CONTEXT_DEPENDENT_TOKENS
    ):
        return False
    specific_tokens = [
        token
        for token in tokens
        if token not in CATEGORY_ANCHOR_TOKENS
        and token not in GENERIC_UTILITY_TOKENS
        and token not in CONTEXT_DEPENDENT_TOKENS
        and token not in FUNCTION_WORDS
    ]
    return bool(specific_tokens)


def selected_family_phrases(row):
    family_map = {}
    for family, column, limit in FAMILY_SPECS:
        values = [
            value for value in split_pipe_values(row.get(column, ""))
            if is_specific_phrase(value)
        ]
        family_map[family] = values[:limit]
    return family_map


def selected_support_phrases(row):
    return [
        value
        for value in split_pipe_values(row.get("common_specific_support_phrases", ""))
        if is_specific_phrase(value)
    ]


def protected_specific_phrases(row):
    phrases = []
    for values in selected_family_phrases(row).values():
        phrases.extend(values)
    phrases.extend(selected_support_phrases(row))
    return list(dict.fromkeys(phrases))


def residual_content_tokens(value):
    blocked = (
        CATEGORY_ANCHOR_TOKENS
        | GENERIC_UTILITY_TOKENS
        | CONTEXT_DEPENDENT_TOKENS
        | RATING_SENTIMENT_TOKENS
        | FUNCTION_WORDS
    )
    out = []
    seen = set()
    for token in tokenize(normalize_query_text(value))[:MAX_RESIDUAL_TOKENS]:
        concept = normalize_content_token(token)
        if token.isdigit() or len(token) < 2 or token in blocked or concept in seen:
            continue
        out.append(token)
        seen.add(concept)
    return out


def build_review_safe_seed(row):
    family_map = selected_family_phrases(row)
    phrases = []
    for family, _, _ in FAMILY_SPECS:
        phrases.extend(family_map[family])
    phrases.extend(selected_support_phrases(row))
    phrases = list(dict.fromkeys(phrases))

    seed_tokens = []
    for phrase in phrases:
        phrase_tokens = tokenize(normalize_query_text(phrase))
        if not phrase_tokens:
            continue
        if len(seed_tokens) + len(phrase_tokens) > MAX_QUERY_TOKENS:
            continue
        seed_tokens.extend(phrase_tokens)

    seen_concepts = {normalize_content_token(token) for token in seed_tokens}
    for token in residual_content_tokens(row.get(RESIDUAL_TEXT_COLUMN, "")):
        if len(seed_tokens) >= RESIDUAL_PAD_TARGET_TOKENS:
            break
        concept = normalize_content_token(token)
        if concept in seen_concepts:
            continue
        seed_tokens.append(token)
        seen_concepts.add(concept)

    return normalize_query_text(" ".join(seed_tokens[:MAX_QUERY_TOKENS]))


def mask_protected_phrases(text, phrases):
    masked = normalize_query_text(text)
    for phrase in sorted(phrases, key=token_count, reverse=True):
        masked = remove_exact_phrase(masked, phrase)
    return masked


def generic_term_audit(text, row):
    masked = mask_protected_phrases(text, protected_specific_phrases(row))
    anchor_hits = []
    for phrase in sorted(CATEGORY_ANCHOR_PHRASES, key=len, reverse=True):
        if exact_phrase_present(masked, phrase):
            anchor_hits.append(phrase)
    anchor_hits.extend(
        token for token in tokenize(masked) if token in CATEGORY_ANCHOR_TOKENS
    )
    utility_hits = [
        token for token in tokenize(masked) if token in GENERIC_UTILITY_TOKENS
    ]
    return pipe_join(anchor_hits), pipe_join(utility_hits)


def query_specific_family_count(text, row):
    return sum(
        any(phrase_supported(text, phrase) for phrase in values)
        for values in selected_family_phrases(row).values()
    )


def missing_seed_multiword_phrases(seed_query, candidate_query, row):
    required = [
        phrase
        for phrase in protected_specific_phrases(row)
        if token_count(phrase) > 1 and exact_phrase_present(seed_query, phrase)
    ]
    return [
        phrase for phrase in required
        if not exact_phrase_present(candidate_query, phrase)
    ]


def prohibited_rating_sentiment_terms(text):
    return pipe_join(
        token for token in tokenize(text) if token in RATING_SENTIMENT_TOKENS
    )


def title_overlap_ratio(query, title):
    query_tokens = set(tokenize(query))
    title_tokens = {token for token in tokenize(title) if len(token) >= 4}
    if not query_tokens or not title_tokens:
        return 0.0
    return len(query_tokens & title_tokens) / len(query_tokens)


def distinctive_metadata_terms(value):
    terms = []
    for phrase in split_metadata_values(value):
        phrase_tokens = tokenize(phrase)
        if len(phrase_tokens) >= 2:
            terms.append(" ".join(phrase_tokens))
        terms.extend(
            token
            for token in phrase_tokens
            if len(token) >= 3 and token not in METADATA_GENERIC_TOKENS
        )
    return terms


def diagnostic_metadata_terms(value):
    terms = []
    for phrase in split_metadata_values(value):
        phrase_tokens = tokenize(phrase)
        if phrase_tokens:
            terms.append(" ".join(phrase_tokens))
        terms.extend(token for token in phrase_tokens if len(token) >= 3)
    return terms


def metadata_term_groups(row):
    name_terms = []
    for column in METADATA_NAME_COLUMNS:
        name_terms.extend(distinctive_metadata_terms(row.get(column, "")))

    identifier_terms = []
    for column in METADATA_DIAGNOSTIC_COLUMNS:
        identifier_terms.extend(diagnostic_metadata_terms(row.get(column, "")))

    title = normalize_query_text(row.get("itemctx_title", ""))
    return {
        "name_terms": list(dict.fromkeys(name_terms)),
        "identifier_terms": list(dict.fromkeys(identifier_terms)),
        "title": title,
    }


def scrub_target_metadata_cues(text, row):
    scrubbed = normalize_query_text(text)
    removed = []

    for label, pattern in [
        ("asin_pattern", ASIN_PATTERN),
        ("package_or_dosage_pattern", PACKAGE_PATTERN),
        ("seller_or_manufacturer_pattern", SELLER_PATTERN),
    ]:
        updated = normalize_query_text(pattern.sub(" ", scrubbed))
        if updated != scrubbed:
            removed.append(label)
        scrubbed = updated

    groups = metadata_term_groups(row)
    for term in groups["name_terms"] + groups["identifier_terms"]:
        updated = remove_exact_phrase(scrubbed, term)
        if updated != scrubbed:
            removed.append(term)
        scrubbed = updated

    title = groups["title"]
    if token_count(title) >= 2:
        updated = remove_exact_phrase(scrubbed, title)
        if updated != scrubbed:
            removed.append("exact_title_phrase")
        scrubbed = updated

    return normalize_query_text(scrubbed), pipe_join(removed)




def repair_seed_from_review_safe_residual(seed, row, minimum_tokens):
    """Pad a post-scrub seed only with safe tokens from the same target review.

    The residual source is Notebook 05 query_safe_text. It is scrubbed against target
    metadata before use. No catalog, prior-history, or population-review evidence is added.
    """
    repaired = normalize_query_text(seed)
    residual_scrubbed, _ = scrub_target_metadata_cues(
        row.get(RESIDUAL_TEXT_COLUMN, ""), row
    )
    seen_concepts = {
        normalize_content_token(token) for token in tokenize(repaired)
    }
    added_tokens = []
    for token in residual_content_tokens(residual_scrubbed):
        if token_count(repaired) >= minimum_tokens:
            break
        concept = normalize_content_token(token)
        if concept in seen_concepts:
            continue
        trial = normalize_query_text(f"{repaired} {token}")
        trial_scrubbed, _ = scrub_target_metadata_cues(trial, row)
        if trial_scrubbed != trial:
            continue
        repaired = trial
        seen_concepts.add(concept)
        added_tokens.append(token)
    return repaired, pipe_join(added_tokens)


def metadata_leakage_flags(text, row):
    normalized = normalize_query_text(text)
    groups = metadata_term_groups(row)
    title = groups["title"]
    return {
        "brand_or_name_leak_flag": int(any(
            exact_phrase_present(normalized, term) for term in groups["name_terms"]
        )),
        "identifier_like_leak_flag": int(any(
            exact_phrase_present(normalized, term) for term in groups["identifier_terms"]
        )),
        "asin_leak_flag": int(bool(ASIN_PATTERN.search(normalized))),
        "package_cue_flag": int(bool(PACKAGE_PATTERN.search(normalized))),
        "seller_manufacturer_leak_flag": int(bool(SELLER_PATTERN.search(normalized))),
        "exact_title_phrase_flag": int(
            token_count(title) >= 2 and exact_phrase_present(normalized, title)
        ),
        "title_overlap_ratio": float(title_overlap_ratio(normalized, title)),
    }


def blocking_leakage_count(flags):
    return sum(
        flags[column]
        for column in [
            "brand_or_name_leak_flag",
            "identifier_like_leak_flag",
            "asin_leak_flag",
            "package_cue_flag",
            "seller_manufacturer_leak_flag",
            "exact_title_phrase_flag",
        ]
    )


def evidence_reason(row):
    reasons = []
    if "signal_available" in row.index and not to_bool(row.get("signal_available")):
        reasons.append("missing_notebook05_signal")
    if to_bool(row.get("signal_available", True)) and not to_bool(
        row.get("review_only_query_evidence_sufficient")
    ):
        reasons.append("upstream_review_only_evidence_insufficient")
    if to_int(row.get("seed_token_count")) < MIN_QUERY_TOKENS:
        reasons.append("review_safe_seed_too_short")
    if to_int(row.get("seed_specific_family_count")) < MIN_SPECIFIC_FAMILIES:
        reasons.append("specific_family_count_below_threshold")
    if to_int(row.get("seed_source_signal_total_count")) < MIN_SPECIFIC_SIGNALS:
        reasons.append("specific_signal_total_below_threshold")
    if normalize_space(row.get("seed_generic_anchor_terms")):
        reasons.append("generic_category_anchor_remaining")
    if normalize_space(row.get("seed_generic_utility_terms")):
        reasons.append("generic_utility_remaining")
    if normalize_space(row.get("seed_rating_sentiment_terms")):
        reasons.append("rating_or_sentiment_term_remaining")
    if to_int(row.get("seed_blocking_leakage_flag_count")) > 0:
        reasons.append("target_metadata_leakage_remaining")
    return " | ".join(reasons)


def bool_series(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).astype(bool)
    return series.fillna("").astype(str).str.strip().str.lower().isin({"true", "1", "yes"})

def eligibility_mask_for_policy(frame, min_tokens, min_families, min_signals):
    upstream_mask = bool_series(frame["review_only_query_evidence_sufficient"])
    safety_mask = (
        frame["seed_generic_anchor_terms"].map(normalize_space).eq("")
        & frame["seed_generic_utility_terms"].map(normalize_space).eq("")
        & frame["seed_rating_sentiment_terms"].map(normalize_space).eq("")
        & pd.to_numeric(
            frame["seed_blocking_leakage_flag_count"], errors="coerce"
        ).fillna(0).eq(0)
    )
    evidence_mask = (
        pd.to_numeric(frame["seed_token_count"], errors="coerce").fillna(0).ge(min_tokens)
        & pd.to_numeric(
            frame["seed_specific_family_count"], errors="coerce"
        ).fillna(0).ge(min_families)
        & pd.to_numeric(
            frame["seed_source_signal_total_count"], errors="coerce"
        ).fillna(0).ge(min_signals)
    )
    return upstream_mask & safety_mask & evidence_mask


In [ ]:
# ==== Load and Validate Cases, Signals, Histories, and Item Universe ====
eligible_columns = [
    "case_id", "user_id", "target_parent_asin", "target_timestamp_ms",
    "target_review_datetime", "target_rank_desc", "target_selection_mode",
    "regime", "selection_rank_within_regime", "initial_order_within_regime",
    "same_regime_replacement_order", "initial_selected", "selection_stage",
]
eligible_pool_df = pd.read_parquet(ELIGIBLE_POOL_PATH, columns=eligible_columns)
review_signal_df = pd.read_parquet(REVIEW_SIGNAL_PATH)
item_docs_df = pd.read_parquet(ITEM_DOCS_PATH, columns=["parent_asin"])

prior_schema_columns = pq.ParquetFile(TRAIN_PRIOR_HISTORY_PATH).schema.names
prior_item_column = "prior_item_id" if "prior_item_id" in prior_schema_columns else "prior_parent_asin"
required_prior_columns = [
    "case_id", "user_id", "target_parent_asin", "target_timestamp_ms", prior_item_column,
]
missing_prior_columns = sorted(set(required_prior_columns) - set(prior_schema_columns))
if missing_prior_columns:
    raise RuntimeError(f"Notebook 03 training prior history is missing columns: {missing_prior_columns}")
train_prior_history_df = pd.read_parquet(
    TRAIN_PRIOR_HISTORY_PATH,
    columns=required_prior_columns,
).rename(columns={prior_item_column: "prior_item_id"})

required_signal_columns = [
    "case_id", "user_id", "target_parent_asin", "regime",
    "query_evidence_source", "item_metadata_evidence_used",
    "historical_review_evidence_used", "user_prior_evidence_used",
    "rating_evidence_used", "sentiment_evidence_used",
    "query_safe_text", "common_specific_support_phrases",
    "common_specific_signal_seed_text", "query_safe_signal_family_count",
    "query_safe_signal_total_count", "review_only_query_evidence_sufficient",
    "flavor_signals",
] + [column for _, column, _ in FAMILY_SPECS]
missing_signal_columns = sorted(set(required_signal_columns) - set(review_signal_df.columns))
if missing_signal_columns:
    raise RuntimeError(f"Notebook 05 output is missing columns: {missing_signal_columns}")

for frame_name, frame in {
    "eligible pool": eligible_pool_df,
    "review signals": review_signal_df,
}.items():
    frame["case_id"] = frame["case_id"].astype(str).str.strip()
    if frame["case_id"].eq("").any() or frame["case_id"].duplicated().any():
        raise RuntimeError(f"{frame_name} must contain unique non-empty case_id values.")

eligible_pool_df["target_parent_asin"] = eligible_pool_df["target_parent_asin"].astype(str).str.strip()
review_signal_df["target_parent_asin"] = review_signal_df["target_parent_asin"].astype(str).str.strip()
eligible_pool_df["target_timestamp_ms"] = pd.to_numeric(
    eligible_pool_df["target_timestamp_ms"], errors="raise"
).astype("int64")
item_docs_df["parent_asin"] = item_docs_df["parent_asin"].astype(str).map(normalize_space)
if item_docs_df["parent_asin"].eq("").any() or item_docs_df["parent_asin"].duplicated().any():
    raise RuntimeError("Notebook 04 item_docs parent_asin keys must be unique and non-empty.")
item_universe_ids = set(item_docs_df["parent_asin"])

for column in ["case_id", "user_id", "target_parent_asin", "prior_item_id"]:
    train_prior_history_df[column] = train_prior_history_df[column].astype(str).map(normalize_space)
train_prior_history_df["target_timestamp_ms"] = pd.to_numeric(
    train_prior_history_df["target_timestamp_ms"], errors="raise"
).astype("int64")

EXPECTED_ELIGIBLE_ROWS = int(len(eligible_pool_df))
if eligible_pool_df["user_id"].duplicated().any():
    raise RuntimeError("Eligible source users must be unique.")

observed_eligible_counts = (
    eligible_pool_df["regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
    .to_dict()
)
EXPECTED_ELIGIBLE_REGIME_COUNTS = dict(observed_eligible_counts)

if any(count <= 0 for count in EXPECTED_ELIGIBLE_REGIME_COUNTS.values()):
    raise RuntimeError(
        f"Notebook 03 source pool must contain at least one case per regime: "
        f"{EXPECTED_ELIGIBLE_REGIME_COUNTS}"
    )
if eligible_pool_df["target_rank_desc"].max() > MAX_TARGET_RANK_ALLOWED:
    raise RuntimeError(f"target_rank_desc must remain <= {MAX_TARGET_RANK_ALLOWED}.")
if not eligible_pool_df["target_selection_mode"].eq(EXPECTED_TARGET_SELECTION_MODE).all():
    raise RuntimeError("target_selection_mode mismatch.")

order_columns = [
    "selection_rank_within_regime",
    "initial_order_within_regime",
    "same_regime_replacement_order",
]
for column in order_columns:
    eligible_pool_df[column] = pd.to_numeric(
        eligible_pool_df[column], errors="raise"
    ).astype(int)
if eligible_pool_df["initial_selected"].notna().any():
    raise RuntimeError("Notebook 03 initial selection must remain deferred to query audit.")
if not eligible_pool_df["selection_stage"].eq("deferred_to_query_audit").all():
    raise RuntimeError("Notebook 03 selection_stage must be deferred_to_query_audit.")
for regime in REGIME_ORDER:
    regime_rows = eligible_pool_df["regime"].eq(regime)
    regime_n = int(regime_rows.sum())
    expected_order = list(range(1, regime_n + 1))
    for column in order_columns:
        observed_order = sorted(
            eligible_pool_df.loc[regime_rows, column].astype(int).tolist()
        )
        if observed_order != expected_order:
            raise RuntimeError(
                f"Notebook 03 {column} is not a contiguous within-regime order for {regime}."
            )

if len(review_signal_df) != EXPECTED_ELIGIBLE_ROWS:
    raise RuntimeError(
        f"Notebook 05 must cover the full Herbal eligible pool: {len(review_signal_df)}/{EXPECTED_ELIGIBLE_ROWS}."
    )
if set(review_signal_df["case_id"]) != set(eligible_pool_df["case_id"]):
    raise RuntimeError("Notebook 05 case coverage must exactly match the Notebook 03 eligible pool.")
if not review_signal_df["query_evidence_source"].eq("target_review_only").all():
    raise RuntimeError("Notebook 05 query_evidence_source must be target_review_only.")
for column in [
    "item_metadata_evidence_used", "historical_review_evidence_used",
    "user_prior_evidence_used", "rating_evidence_used", "sentiment_evidence_used",
]:
    if bool_series(review_signal_df[column]).any():
        raise RuntimeError(f"Notebook 05 evidence flag must be false: {column}")

forbidden_signal_columns = {
    "target_review_text", "heldout_review_text", "review_text", "review_body",
    "raw_review_text", "prior_review_text", "prior_history_n", "prior_review_n",
    "rating", "sentiment", "prompt", "response", "llm_response",
    "query_safe_facet_text", "common_functional_facet_text",
    "historical_review_reputation_text", "review_reputation_facet_text",
}
forbidden_signal_columns_present = sorted(forbidden_signal_columns & set(review_signal_df.columns))
if forbidden_signal_columns_present:
    raise RuntimeError(
        f"Forbidden query-evidence columns are present in Notebook 05 output: {forbidden_signal_columns_present}"
    )

source_df = eligible_pool_df.merge(
    review_signal_df,
    on=["case_id", "user_id", "target_parent_asin", "regime"],
    how="left",
    validate="one_to_one",
    suffixes=("", "_signal"),
)
if source_df["query_safe_text"].isna().any():
    raise RuntimeError("Notebook 05 signal join did not match every eligible case.")

source_case_ids = set(source_df["case_id"])
train_prior_history_df = train_prior_history_df[
    train_prior_history_df["case_id"].isin(source_case_ids)
    & train_prior_history_df["prior_item_id"].ne("")
].copy()
if len(train_prior_history_df):
    source_user_by_case = source_df.set_index("case_id")["user_id"]
    source_target_by_case = source_df.set_index("case_id")["target_parent_asin"]
    source_timestamp_by_case = source_df.set_index("case_id")["target_timestamp_ms"]
    if not train_prior_history_df["user_id"].eq(train_prior_history_df["case_id"].map(source_user_by_case)).all():
        raise RuntimeError("Notebook 03 training prior-history user_id does not match the eligible case.")
    if not train_prior_history_df["target_parent_asin"].eq(train_prior_history_df["case_id"].map(source_target_by_case)).all():
        raise RuntimeError("Notebook 03 training prior-history target item does not match the eligible case.")
    if not train_prior_history_df["target_timestamp_ms"].eq(train_prior_history_df["case_id"].map(source_timestamp_by_case)).all():
        raise RuntimeError("Notebook 03 training prior-history target timestamp does not match the eligible case.")
    if train_prior_history_df["prior_item_id"].eq(train_prior_history_df["case_id"].map(source_target_by_case)).any():
        raise RuntimeError("Notebook 03 training prior history contains a held-out target item.")

train_prior_in_item_universe_df = train_prior_history_df[
    train_prior_history_df["prior_item_id"].isin(item_universe_ids)
].copy()
training_prior_item_universe_counts = train_prior_in_item_universe_df.groupby("case_id")["prior_item_id"].nunique()
source_df["training_prior_item_in_item_universe_count"] = (
    source_df["case_id"].map(training_prior_item_universe_counts).fillna(0).astype(int)
)
source_df["downstream_personalized_prior_item_available"] = (
    source_df["regime"].eq("cold")
    | source_df["training_prior_item_in_item_universe_count"].gt(0)
)

cold_prior_item_mismatch = source_df[
    source_df["regime"].eq("cold")
    & source_df["training_prior_item_in_item_universe_count"].ne(0)
].copy()
if len(cold_prior_item_mismatch):
    display(cold_prior_item_mismatch.head(20))
    raise RuntimeError("Cold cases must not have training-safe prior items in the Notebook 04 item universe.")

print("Rows: eligible", len(eligible_pool_df))
print("Rows: signals", len(review_signal_df))
print("Rows: training prior history", len(train_prior_history_df))
print("Rows: training prior history in Notebook 04 item universe", len(train_prior_in_item_universe_df))
print("Validation: upstream contracts passed")

In [ ]:
# ==== Build Review-Only Seeds Before the Target-Metadata Audit ====
seed_records = []
for row in source_df.itertuples(index=False):
    row_dict = row._asdict()
    seed_records.append({
        "case_id": str(row_dict["case_id"]),
        "review_safe_seed_before_metadata_audit": build_review_safe_seed(row_dict),
    })
source_seed_df = source_df.merge(pd.DataFrame(seed_records), on="case_id", how="left", validate="one_to_one")

item_context_df = pd.read_parquet(
    ITEM_SCHEMA_PATH,
    columns=[
        "parent_asin", "title", "facet_brand_text", "manufacturer", "store",
        "package_dimensions", "unit_count", "number_of_items", "item_model_number",
        "identifier_diagnostic_text",
    ],
)
item_context_df["parent_asin"] = item_context_df["parent_asin"].astype(str).str.strip()
if item_context_df["parent_asin"].eq("").any() or item_context_df["parent_asin"].duplicated().any():
    raise RuntimeError("Item schema audit keys must be unique and non-empty.")
item_context_df = item_context_df.rename(columns={
    "parent_asin": "target_parent_asin",
    "title": "itemctx_title",
    "facet_brand_text": "itemctx_facet_brand_text",
    "manufacturer": "itemctx_manufacturer",
    "store": "itemctx_store",
    "package_dimensions": "itemctx_package_dimensions",
    "unit_count": "itemctx_unit_count",
    "number_of_items": "itemctx_number_of_items",
    "item_model_number": "itemctx_item_model_number",
    "identifier_diagnostic_text": "itemctx_identifier_diagnostic_text",
})

audit_df = source_seed_df.merge(
    item_context_df,
    on="target_parent_asin",
    how="left",
    validate="many_to_one",
    indicator=True,
)
if not audit_df["_merge"].eq("both").all():
    raise RuntimeError("Target metadata audit join did not match every eligible case.")
audit_df = audit_df.drop(columns="_merge")
for column in [
    "itemctx_title", "itemctx_facet_brand_text", "itemctx_manufacturer", "itemctx_store",
    "itemctx_package_dimensions", "itemctx_unit_count", "itemctx_number_of_items",
    "itemctx_item_model_number", "itemctx_identifier_diagnostic_text",
]:
    audit_df[column] = audit_df[column].fillna("").astype(str)

candidate_records = []
for row in audit_df.itertuples(index=False):
    row_dict = row._asdict()
    scrubbed_seed_before_repair, removed_terms_before_repair = scrub_target_metadata_cues(
        row_dict["review_safe_seed_before_metadata_audit"], row_dict
    )
    scrubbed_seed, residual_repair_tokens = repair_seed_from_review_safe_residual(
        scrubbed_seed_before_repair, row_dict, MIN_QUERY_TOKENS
    )
    scrubbed_seed, removed_terms_after_repair = scrub_target_metadata_cues(
        scrubbed_seed, row_dict
    )
    removed_terms = pipe_join(
        split_pipe_values(removed_terms_before_repair)
        + split_pipe_values(removed_terms_after_repair)
    )
    leak_flags = metadata_leakage_flags(scrubbed_seed, row_dict)
    generic_anchors, generic_utilities = generic_term_audit(scrubbed_seed, row_dict)
    family_count = query_specific_family_count(scrubbed_seed, row_dict)
    rating_sentiment_terms = prohibited_rating_sentiment_terms(scrubbed_seed)
    blocking_count = blocking_leakage_count(leak_flags)
    source_signal_total_count = to_int(row_dict.get("query_safe_signal_total_count"))
    sufficient = bool(
        to_bool(row_dict.get("review_only_query_evidence_sufficient"))
        and token_count(scrubbed_seed) >= MIN_QUERY_TOKENS
        and family_count >= MIN_SPECIFIC_FAMILIES
        and source_signal_total_count >= MIN_SPECIFIC_SIGNALS
        and not generic_anchors
        and not generic_utilities
        and not rating_sentiment_terms
        and blocking_count == 0
    )
    candidate_records.append({
        "case_id": str(row_dict["case_id"]),
        "review_safe_seed": scrubbed_seed,
        "seed_token_count_before_residual_repair": token_count(scrubbed_seed_before_repair),
        "seed_token_count": token_count(scrubbed_seed),
        "seed_residual_repair_tokens": residual_repair_tokens,
        "seed_residual_repair_token_count": len(split_pipe_values(residual_repair_tokens)),
        "seed_specific_family_count": int(family_count),
        "seed_source_signal_total_count": int(source_signal_total_count),
        "seed_generic_anchor_terms": generic_anchors,
        "seed_generic_utility_terms": generic_utilities,
        "seed_rating_sentiment_terms": rating_sentiment_terms,
        "seed_metadata_removed_terms": removed_terms,
        "seed_blocking_leakage_flag_count": int(blocking_count),
        "seed_title_overlap_ratio": float(leak_flags["title_overlap_ratio"]),
        "query_candidate_sufficient": sufficient,
    })

candidate_df = audit_df.merge(
    pd.DataFrame(candidate_records), on="case_id", how="left", validate="one_to_one"
)
candidate_df["insufficient_reason"] = candidate_df.apply(evidence_reason, axis=1)

print("Rows: sufficient under active policy", int(candidate_df["query_candidate_sufficient"].sum()))
print("Rows: residual-safe repair used", int(candidate_df["seed_residual_repair_token_count"].gt(0).sum()))
print("Validation: review-only seed audit passed")

In [ ]:
# ==== Audit Eligibility, Fix the Balanced Quota, and Map Replacements ====
# Determine eligibility and the balanced quota before any DSPy call.
# In identity-preflight mode, canonical query artifacts remain unwritten.
preflight_publication_enabled = bool(ALLOW_CANONICAL_PUBLICATION) and not bool(RUN_IDENTITY_ONLY_PREFLIGHT)
if preflight_publication_enabled:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

active_policy_mask = eligibility_mask_for_policy(candidate_df, **ACTIVE_ELIGIBILITY_POLICY)
if not active_policy_mask.equals(candidate_df["query_candidate_sufficient"].astype(bool)):
    mismatch_count = int(
        active_policy_mask.ne(candidate_df["query_candidate_sufficient"].astype(bool)).sum()
    )
    raise RuntimeError(
        "Active eligibility policy and query_candidate_sufficient disagree: "
        f"{mismatch_count} rows."
    )

active_feasible_counts = (
    candidate_df.loc[active_policy_mask, "regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
)
downstream_prior_guard_mask = candidate_df["downstream_personalized_prior_item_available"].astype(bool)
selection_eligible_mask = active_policy_mask & downstream_prior_guard_mask
candidate_df["downstream_prior_item_guard_passed"] = downstream_prior_guard_mask
candidate_df["selection_eligible"] = selection_eligible_mask
candidate_df["selection_insufficient_reason"] = candidate_df["insufficient_reason"].fillna("").astype(str)
missing_downstream_prior_mask = active_policy_mask & ~downstream_prior_guard_mask
candidate_df.loc[missing_downstream_prior_mask, "selection_insufficient_reason"] = candidate_df.loc[
    missing_downstream_prior_mask, "selection_insufficient_reason"
].map(
    lambda value: pipe_join(
        split_pipe_values(value) + ["non_cold_training_prior_item_outside_item_universe"]
    )
)
selection_feasible_counts = (
    candidate_df.loc[selection_eligible_mask, "regime"]
    .value_counts()
    .reindex(REGIME_ORDER, fill_value=0)
    .astype(int)
)
EXPECTED_TARGET_PER_REGIME = int(selection_feasible_counts.min())
EXPECTED_OUTPUT_ROWS = int(EXPECTED_TARGET_PER_REGIME * len(REGIME_ORDER))

# Trace cumulative attrition from Notebook 05 evidence sufficiency through
# the active family, length, language-safety, and metadata-safety requirements.
all_rows_mask = pd.Series(True, index=candidate_df.index, dtype=bool)
upstream_sufficient_mask = candidate_df["review_only_query_evidence_sufficient"].map(to_bool)
source_signal_mask = pd.to_numeric(
    candidate_df["seed_source_signal_total_count"], errors="coerce"
).fillna(0).ge(MIN_SPECIFIC_SIGNALS)
active_family_mask = pd.to_numeric(
    candidate_df["seed_specific_family_count"], errors="coerce"
).fillna(0).ge(MIN_SPECIFIC_FAMILIES)
minimum_length_mask = pd.to_numeric(
    candidate_df["seed_token_count"], errors="coerce"
).fillna(0).ge(MIN_QUERY_TOKENS)
language_safety_mask = (
    candidate_df["seed_generic_anchor_terms"].fillna("").astype(str).str.strip().eq("")
    & candidate_df["seed_generic_utility_terms"].fillna("").astype(str).str.strip().eq("")
    & candidate_df["seed_rating_sentiment_terms"].fillna("").astype(str).str.strip().eq("")
)
metadata_safety_mask = pd.to_numeric(
    candidate_df["seed_blocking_leakage_flag_count"], errors="coerce"
).fillna(0).eq(0)

attrition_steps = [
    ("source_pool", all_rows_mask),
    ("notebook05_review_only_evidence_sufficient", upstream_sufficient_mask),
    (
        "source_signal_available",
        upstream_sufficient_mask & source_signal_mask,
    ),
    (
        "active_nonflavor_functional_family_available",
        upstream_sufficient_mask & source_signal_mask & active_family_mask,
    ),
    (
        "safe_seed_at_least_5_tokens",
        upstream_sufficient_mask
        & source_signal_mask
        & active_family_mask
        & minimum_length_mask,
    ),
    (
        "no_generic_rating_or_sentiment_terms",
        upstream_sufficient_mask
        & source_signal_mask
        & active_family_mask
        & minimum_length_mask
        & language_safety_mask,
    ),
    (
        "no_blocking_metadata_leakage",
        upstream_sufficient_mask
        & source_signal_mask
        & active_family_mask
        & minimum_length_mask
        & language_safety_mask
        & metadata_safety_mask,
    ),
    ("final_active_policy", active_policy_mask),
    ("downstream_non_cold_prior_item_in_item_universe", selection_eligible_mask),
]

attrition_rows = []
previous_counts = {regime: None for regime in REGIME_ORDER}
for step_index, (step_name, step_mask) in enumerate(attrition_steps):
    step_counts = (
        candidate_df.loc[step_mask, "regime"]
        .value_counts()
        .reindex(REGIME_ORDER, fill_value=0)
        .astype(int)
    )
    for regime in REGIME_ORDER:
        retained_cases = int(step_counts.loc[regime])
        previous_count = previous_counts[regime]
        attrition_rows.append({
            "step_index": int(step_index),
            "step_name": step_name,
            "regime": regime,
            "retained_cases": retained_cases,
            "lost_from_previous_step": (
                0 if previous_count is None else int(previous_count - retained_cases)
            ),
            "retention_rate_from_source": float(
                retained_cases / max(int(candidate_df["regime"].eq(regime).sum()), 1)
            ),
        })
        previous_counts[regime] = retained_cases

pre_generation_attrition_qc_df = pd.DataFrame(attrition_rows)

failure_reason_summary_df = (
    candidate_df.loc[~selection_eligible_mask, ["regime", "selection_insufficient_reason"]]
    .assign(selection_insufficient_reason=lambda frame: frame["selection_insufficient_reason"].replace("", "unclassified"))
    .assign(selection_insufficient_reason=lambda frame: frame["selection_insufficient_reason"].str.split(r"\s*\|\s*"))
    .explode("selection_insufficient_reason")
    .rename(columns={"selection_insufficient_reason": "insufficient_reason"})
    .groupby(["regime", "insufficient_reason"], dropna=False)
    .size()
    .rename("case_count")
    .reset_index()
    .sort_values(["regime", "case_count", "insufficient_reason"], ascending=[True, False, True])
)

if preflight_publication_enabled:
    failure_reason_summary_df.to_csv(
        PRE_GENERATION_FAILURE_REASON_PATH, index=False, encoding="utf-8-sig"
    )
    pre_generation_attrition_qc_df.to_csv(
        PRE_GENERATION_ATTRITION_QC_PATH, index=False, encoding="utf-8-sig"
    )

if EXPECTED_TARGET_PER_REGIME <= 0:
    raise RuntimeError(
        "No balanced review-only query quota is feasible under the active policy and downstream prior-item guard: "
        f"policy={ELIGIBILITY_POLICY}, counts={selection_feasible_counts.to_dict()}"
    )

print("Cumulative pre-generation attrition:")
display(
    pre_generation_attrition_qc_df.pivot_table(
        index=["step_index", "step_name"],
        columns="regime",
        values="retained_cases",
        aggfunc="first",
    ).reset_index().sort_values("step_index")
)
print("Active eligibility policy:", ELIGIBILITY_POLICY)
print("Active feasible counts:", active_feasible_counts.to_dict())
print("Selection feasible counts:", selection_feasible_counts.to_dict())
print("Dynamic balanced target per regime:", EXPECTED_TARGET_PER_REGIME)
print("No DSPy call has occurred before this QC.")

ranked_candidate_df = candidate_df.copy()
ranked_candidate_df["_regime_order"] = ranked_candidate_df["regime"].map(
    {regime: index for index, regime in enumerate(REGIME_ORDER)}
)
if ranked_candidate_df["_regime_order"].isna().any():
    raise RuntimeError("Candidate pool contains an unexpected regime.")
ranked_candidate_df = (
    ranked_candidate_df.sort_values(
        ["_regime_order", "initial_order_within_regime", "case_id"],
        kind="mergesort",
    )
    .drop(columns="_regime_order")
    .reset_index(drop=True)
)
ranked_candidate_df["initial_selected"] = ranked_candidate_df[
    "initial_order_within_regime"
].le(EXPECTED_TARGET_PER_REGIME)

insufficient_audit_df = ranked_candidate_df.loc[
    ~ranked_candidate_df["selection_eligible"],
    [
        "case_id", "user_id", "target_parent_asin", "regime", "target_rank_desc",
        "selection_rank_within_regime", "initial_order_within_regime",
        "same_regime_replacement_order", "initial_selected", "selection_eligible",
        "downstream_prior_item_guard_passed", "training_prior_item_in_item_universe_count",
        "review_only_query_evidence_sufficient", "query_safe_signal_family_count",
        "query_safe_signal_total_count", "seed_token_count_before_residual_repair",
        "seed_token_count", "seed_residual_repair_token_count",
        "seed_residual_repair_tokens", "seed_specific_family_count",
        "seed_generic_anchor_terms", "seed_generic_utility_terms",
        "seed_rating_sentiment_terms", "seed_blocking_leakage_flag_count",
        "insufficient_reason", "selection_insufficient_reason",
    ],
].copy()
if preflight_publication_enabled:
    insufficient_audit_df.to_csv(
        OUTPUT_DIR / "herbal_query_insufficient_review_evidence.csv",
        index=False,
        encoding="utf-8-sig",
    )

selection_plan_rows = []
replacement_rows = []
shortfalls = {}

for regime in REGIME_ORDER:
    regime_ranked = ranked_candidate_df[ranked_candidate_df["regime"].eq(regime)].copy()
    regime_ranked = regime_ranked.sort_values(["initial_order_within_regime", "case_id"], kind="mergesort")
    initial_regime = regime_ranked[regime_ranked["initial_selected"]].copy()
    sufficient_regime = regime_ranked[regime_ranked["selection_eligible"]].copy()

    if len(sufficient_regime) < EXPECTED_TARGET_PER_REGIME:
        shortfalls[regime] = EXPECTED_TARGET_PER_REGIME - len(sufficient_regime)

    replacement_candidates = sufficient_regime[
        ~sufficient_regime["initial_selected"]
    ].sort_values(["same_regime_replacement_order", "case_id"], kind="mergesort")
    replacement_case_ids = replacement_candidates["case_id"].tolist()
    replacement_index = 0

    for initial_row in initial_regime.itertuples(index=False):
        initial_case_id = str(initial_row.case_id)
        if bool(initial_row.selection_eligible):
            final_case_id = initial_case_id
            replacement_used = False
            replacement_source_case_id = ""
            replaced_case_id = ""
        elif replacement_index < len(replacement_case_ids):
            final_case_id = replacement_case_ids[replacement_index]
            replacement_index += 1
            replacement_used = True
            replacement_source_case_id = final_case_id
            replaced_case_id = initial_case_id
            replacement_rows.append({
                "regime": regime,
                "replaced_case_id": initial_case_id,
                "replacement_source_case_id": final_case_id,
                "replacement_status": "replaced_from_same_regime_reserve",
            })
        else:
            final_case_id = ""
            replacement_used = True
            replacement_source_case_id = final_case_id
            replaced_case_id = initial_case_id
            replacement_rows.append({
                "regime": regime,
                "replaced_case_id": initial_case_id,
                "replacement_source_case_id": "",
                "replacement_status": "reserve_shortfall",
            })

        selection_plan_rows.append({
            "regime": regime,
            "selection_slot": int(initial_row.initial_order_within_regime),
            "initial_case_id": initial_case_id,
            "final_case_id": final_case_id,
            "replacement_case_used": bool(replacement_used),
            "replacement_source_case_id": replacement_source_case_id,
            "replaced_case_id": replaced_case_id,
        })

replacement_mapping_df = pd.DataFrame(replacement_rows, columns=[
    "regime", "replaced_case_id", "replacement_source_case_id", "replacement_status"
])
if preflight_publication_enabled:
    replacement_mapping_df.to_csv(
        OUTPUT_DIR / "herbal_query_replacement_mapping.csv",
        index=False,
        encoding="utf-8-sig",
    )

if shortfalls:
    raise RuntimeError(
        "Same-regime reserve cannot satisfy the active review-only quota. "
        f"Shortfalls: {shortfalls}. Target metadata and generic-anchor fallback are prohibited."
    )

selection_plan_df = pd.DataFrame(selection_plan_rows)
if selection_plan_df["final_case_id"].eq("").any():
    raise RuntimeError("Final selection plan contains an unfilled query slot.")
if selection_plan_df["final_case_id"].duplicated().any():
    raise RuntimeError("Replacement selection produced duplicate final case ids.")

final_selection_df = selection_plan_df.merge(
    ranked_candidate_df,
    left_on="final_case_id",
    right_on="case_id",
    how="left",
    validate="one_to_one",
)
if not final_selection_df["query_candidate_sufficient"].all():
    raise RuntimeError("Final selection contains insufficient review-only evidence.")
if not final_selection_df["downstream_prior_item_guard_passed"].all():
    raise RuntimeError("Final selection contains a non-cold case without a training-safe prior item in the Notebook 04 item universe.")
if not final_selection_df["selection_eligible"].all():
    raise RuntimeError("Final selection contains a case that is not selection eligible.")
if not final_selection_df["regime_x"].eq(final_selection_df["regime_y"]).all():
    raise RuntimeError("Replacement crossed regime boundaries.")
if final_selection_df["user_id"].duplicated().any():
    raise RuntimeError("Final replacement selection must retain unique users.")

final_counts = (
    final_selection_df["regime_x"].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int)
)
if not final_counts.eq(EXPECTED_TARGET_PER_REGIME).all():
    raise RuntimeError(f"Final replacement quota mismatch: {final_counts.to_dict()}")

print("Rows: insufficient", len(insufficient_audit_df))
identity_preflight_rows = [
    {
        "check_name": "notebook03_initial_slots_from_locked_initial_order",
        "passed": bool(
            selection_plan_df["selection_slot"].astype(int).between(1, EXPECTED_TARGET_PER_REGIME).all()
        ),
        "observed": int(selection_plan_df["selection_slot"].max()),
        "expected": int(EXPECTED_TARGET_PER_REGIME),
    },
    {
        "check_name": "same_regime_replacement_order_locked",
        "passed": bool(
            replacement_mapping_df.empty
            or replacement_mapping_df["replacement_status"].eq("replaced_from_same_regime_reserve").all()
        ),
        "observed": sorted(replacement_mapping_df["replacement_status"].unique().tolist()),
        "expected": ["replaced_from_same_regime_reserve"],
    },
    {
        "check_name": "replacement_lineage_exported",
        "passed": set(["regime", "replaced_case_id", "replacement_source_case_id", "replacement_status"]).issubset(replacement_mapping_df.columns),
        "observed": list(replacement_mapping_df.columns),
        "expected": ["regime", "replaced_case_id", "replacement_source_case_id", "replacement_status"],
    },
    {
        "check_name": "final_quota_regime_preserving",
        "passed": bool(final_counts.eq(EXPECTED_TARGET_PER_REGIME).all()),
        "observed": final_counts.to_dict(),
        "expected": {regime: int(EXPECTED_TARGET_PER_REGIME) for regime in REGIME_ORDER},
    },
]
identity_preflight_qc_df = pd.DataFrame(identity_preflight_rows)
IDENTITY_PREFLIGHT_PASSED = bool(identity_preflight_qc_df["passed"].all())
if not IDENTITY_PREFLIGHT_PASSED:
    raise RuntimeError("SC-0 locked-order identity preflight failed.")
if RUN_IDENTITY_ONLY_PREFLIGHT:
    USE_DSPY_LINGUISTIC_REWRITE = False
    FAIL_IF_DSPY_UNAVAILABLE = False

print("Rows: replacements", len(replacement_mapping_df))
print("Rows: final", len(final_selection_df))
print("Validation: replacement contract passed")
print("SC-0 identity preflight: PASS")


In [ ]:
# ==== Apply the Constrained DSPy Rewrite with Deterministic Fallback ====
def get_deepseek_api_key():
    from google.colab import userdata
    return userdata.get(DEEPSEEK_COLAB_SECRET)


def configure_dspy(api_key):
    dspy.configure(
        lm=dspy.LM(
            DEEPSEEK_MODEL,
            api_key=api_key,
            api_base=DEEPSEEK_BASE_URL,
            temperature=DSPY_TEMPERATURE,
            max_tokens=DSPY_MAX_TOKENS,
        )
    )

    class HerbalQueryLinguisticRewrite(dspy.Signature):
        """Rewrite a retrieval query without adding new product attributes.

        Use only the deterministic seed query. Do not add herbs, ingredients,
        benefits, forms, claims, flavors, brands, product names, package sizes,
        dosages, identifiers, ratings, or sentiment terms. Only reorder, lightly
        grammaticalize, or add function words.
        """

        seed_query = dspy.InputField(
            desc="Deterministic review-safe seed query and only evidence source."
        )
        query_c = dspy.OutputField(
            desc="Concise retrieval query using only seed-query content."
        )

    return dspy.Predict(HerbalQueryLinguisticRewrite)


DSPY_PROGRAM = None
if USE_DSPY_LINGUISTIC_REWRITE:
    api_key = get_deepseek_api_key()
    if not api_key and FAIL_IF_DSPY_UNAVAILABLE:
        raise RuntimeError(
            f"DeepSeek API key is required in Colab secret {DEEPSEEK_COLAB_SECRET!r}."
        )
    if api_key:
        DSPY_PROGRAM = configure_dspy(api_key)


def call_dspy(seed_query):
    if not USE_DSPY_LINGUISTIC_REWRITE:
        return seed_query, "dspy_disabled"
    if DSPY_PROGRAM is None:
        return seed_query, "dspy_unavailable"
    for attempt in range(1, DSPY_MAX_RETRIES + 1):
        prediction = DSPY_PROGRAM(seed_query=seed_query)
        rewritten = normalize_query_text(getattr(prediction, "query_c", ""))
        if rewritten:
            return rewritten, "dspy_linguistic_rewrite"
        if attempt < DSPY_MAX_RETRIES:
            time.sleep(DSPY_RETRY_SLEEP_SECONDS)
    return seed_query, "dspy_empty_output"


def rewrite_new_content_tokens(seed_query, candidate_query):
    return sorted(content_tokens(candidate_query) - content_tokens(seed_query))


def validate_generated_query(query, seed_query, row):
    scrubbed, removed_terms = scrub_target_metadata_cues(query, row)
    leak_flags = metadata_leakage_flags(scrubbed, row)
    generic_anchors, generic_utilities = generic_term_audit(scrubbed, row)
    family_count = query_specific_family_count(scrubbed, row)
    new_tokens = rewrite_new_content_tokens(seed_query, scrubbed)
    missing_multiword = missing_seed_multiword_phrases(seed_query, scrubbed, row)
    rating_sentiment_terms = prohibited_rating_sentiment_terms(scrubbed)
    valid = bool(
        MIN_QUERY_TOKENS <= token_count(scrubbed) <= MAX_QUERY_TOKENS
        and family_count >= MIN_SPECIFIC_FAMILIES
        and not generic_anchors
        and not generic_utilities
        and not rating_sentiment_terms
        and not new_tokens
        and not missing_multiword
        and blocking_leakage_count(leak_flags) == 0
    )
    return {
        "query": scrubbed,
        "valid": valid,
        "removed_terms": removed_terms,
        "generic_anchor_terms": generic_anchors,
        "generic_utility_terms": generic_utilities,
        "rating_sentiment_terms": rating_sentiment_terms,
        "specific_family_count": int(family_count),
        "new_content_tokens": new_tokens,
        "missing_multiword_phrases": missing_multiword,
        **leak_flags,
    }


def herbal_locked_order_preflight_sample(frame, smoke_n):
    if smoke_n <= 0:
        raise RuntimeError("SMOKE_TEST_N must be positive.")
    ordered = frame.sort_values(["regime_x", "selection_slot", "case_id"], kind="mergesort")
    base_n, remainder = divmod(smoke_n, len(REGIME_ORDER))
    parts = []
    for index, regime in enumerate(REGIME_ORDER):
        regime_n = base_n + int(index < remainder)
        if regime_n <= 0:
            continue
        regime_df = ordered[ordered["regime_x"].eq(regime)].copy()
        parts.append(regime_df.head(min(regime_n, len(regime_df))))
    selected = pd.concat(parts, ignore_index=True) if parts else ordered.head(0).copy()
    if len(selected) < smoke_n:
        remaining = ordered[~ordered["case_id"].isin(selected["case_id"])].copy()
        selected = pd.concat(
            [selected, remaining.head(min(smoke_n - len(selected), len(remaining)))],
            ignore_index=True,
        )
    return selected.head(smoke_n).copy()

generation_df = (
    herbal_locked_order_preflight_sample(final_selection_df, SMOKE_TEST_N)
    if RUN_SMOKE_TEST
    else final_selection_df.sort_values(["regime_x", "case_id"]).reset_index(drop=True)
)

query_rows = []
qc_rows = []
for row in generation_df.itertuples(index=False):
    row_dict = row._asdict()
    seed_query = normalize_query_text(row_dict["review_safe_seed"])
    candidate_query, dspy_status = call_dspy(seed_query)
    candidate_audit = validate_generated_query(candidate_query, seed_query, row_dict)

    if candidate_audit["valid"]:
        final_audit = candidate_audit
        dspy_accepted = dspy_status == "dspy_linguistic_rewrite"
        query_source = "dspy_linguistic_rewrite" if dspy_accepted else "deterministic_review_safe_seed"
        fallback_used = not dspy_accepted
    else:
        final_audit = validate_generated_query(seed_query, seed_query, row_dict)
        if not final_audit["valid"]:
            raise RuntimeError(
                f"Prevalidated deterministic seed failed final validation for case {row_dict['case_id']}."
            )
        dspy_accepted = False
        query_source = "deterministic_review_safe_seed_fallback"
        fallback_used = True

    final_query = final_audit["query"]
    anchor_term_count = len(split_pipe_values(final_audit["generic_anchor_terms"]))

    query_row = {
        "case_id": str(row_dict["case_id"]),
        "user_id": str(row_dict["user_id"]),
        "regime": str(row_dict["regime_x"]),
        "target_parent_asin": str(row_dict["target_parent_asin"]),
        "parent_asin": str(row_dict["target_parent_asin"]),
        "target_timestamp_ms": int(row_dict["target_timestamp_ms"]),
        "target_review_datetime": row_dict["target_review_datetime"],
        "target_rank_desc": int(row_dict["target_rank_desc"]),
        "target_selection_mode": str(row_dict["target_selection_mode"]),
        "safe_signal_count": to_int(row_dict.get("query_safe_signal_total_count")),
        "query": final_query,
        "query_C": final_query,
        "query_seed": seed_query,
        "query_source": query_source,
        "query_variant": QUERY_VARIANT,
        "query_dspy_accepted": bool(dspy_accepted),
        "query_dspy_status": dspy_status,
        "query_fallback_used": bool(fallback_used),
        "query_new_content_tokens": pipe_join(final_audit["new_content_tokens"]),
        "query_new_content_token_count": len(final_audit["new_content_tokens"]),
        "query_evidence_source": "target_review_safe_signals_only",
        "target_metadata_fallback_used": False,
        "item_context_fallback_used": False,
        "item_metadata_evidence_used": False,
        "historical_review_evidence_used": False,
        "user_prior_evidence_used": False,
        "rating_evidence_used": False,
        "sentiment_evidence_used": False,
        "insufficient_review_evidence": False,
        "replacement_case_used": bool(row_dict["replacement_case_used"]),
        "initial_case_id": str(row_dict["initial_case_id"]),
        "replaced_case_id": str(row_dict["replaced_case_id"]),
        "replacement_source_case_id": str(row_dict["replacement_source_case_id"]),
        "query_generation_status": "generated_from_review_safe_signals",
        "query_specific_facet_family_count": int(final_audit["specific_family_count"]),
        "query_generic_anchor_terms": final_audit["generic_anchor_terms"],
        "query_generic_utility_terms": final_audit["generic_utility_terms"],
        "query_generic_anchor_rate": float(anchor_term_count / max(token_count(final_query), 1)),
        "query_specific_facet_cue_rate": float(final_audit["specific_family_count"] > 0),
        "query_clean": final_query,
        "query_clean_is_active": False,
    }
    query_rows.append(query_row)

    qc_rows.append({
        **query_row,
        "query_token_count": token_count(final_query),
        "query_seed_token_count": token_count(seed_query),
        "query_metadata_removed_terms": final_audit["removed_terms"],
        "brand_or_name_leak_flag": int(final_audit["brand_or_name_leak_flag"]),
        "identifier_like_leak_flag": int(final_audit["identifier_like_leak_flag"]),
        "asin_leak_flag": int(final_audit["asin_leak_flag"]),
        "package_cue_flag": int(final_audit["package_cue_flag"]),
        "seller_manufacturer_leak_flag": int(final_audit["seller_manufacturer_leak_flag"]),
        "exact_title_phrase_flag": int(final_audit["exact_title_phrase_flag"]),
        "query_rating_sentiment_terms": final_audit["rating_sentiment_terms"],
        "query_rating_sentiment_term_count": len(split_pipe_values(final_audit["rating_sentiment_terms"])),
        "query_missing_multiword_phrases": pipe_join(final_audit["missing_multiword_phrases"]),
        "query_missing_multiword_phrase_count": len(final_audit["missing_multiword_phrases"]),
        "title_overlap_ratio": float(final_audit["title_overlap_ratio"]),
        "title_overlap_flag": int(final_audit["title_overlap_ratio"] > TITLE_OVERLAP_WARNING_THRESHOLD),
    })

queries_df = pd.DataFrame(query_rows)
query_qc_df = pd.DataFrame(qc_rows)

print("Rows: generated", len(queries_df))
print("Validation: query generation completed")

In [ ]:
# ==== Validate the Final Query Contract and Assemble Summaries ====
EXPECTED_OUTPUT_ROWS = int(EXPECTED_TARGET_PER_REGIME * len(REGIME_ORDER))
expected_rows = SMOKE_TEST_N if RUN_SMOKE_TEST else EXPECTED_OUTPUT_ROWS
if len(queries_df) != expected_rows:
    raise RuntimeError(f"Expected {expected_rows} query rows, found {len(queries_df)}.")
if queries_df["case_id"].duplicated().any():
    raise RuntimeError("case_id must be unique in the final query cache.")
if queries_df["query"].fillna("").astype(str).str.strip().eq("").any():
    raise RuntimeError("Every active query must be non-empty.")
if not queries_df["query_C"].eq(queries_df["query"]).all():
    raise RuntimeError("query_C must be an exact compatibility alias of query.")

if not RUN_SMOKE_TEST:
    final_regime_counts = (
        queries_df["regime"].value_counts().reindex(REGIME_ORDER, fill_value=0).astype(int)
    )
    if not final_regime_counts.eq(EXPECTED_TARGET_PER_REGIME).all():
        raise RuntimeError(f"Final regime quota mismatch: {final_regime_counts.to_dict()}")

required_query_audit_columns = {
    "query_evidence_source", "target_metadata_fallback_used",
    "item_context_fallback_used", "historical_review_evidence_used",
    "user_prior_evidence_used", "insufficient_review_evidence",
    "replacement_case_used", "replacement_source_case_id",
    "query_generation_status", "query_specific_facet_family_count",
    "query_generic_anchor_terms", "query_generic_utility_terms",
}
missing_query_audit_columns = sorted(required_query_audit_columns - set(queries_df.columns))
if missing_query_audit_columns:
    raise RuntimeError(f"Missing required query audit columns: {missing_query_audit_columns}")

required_false_columns = [
    "target_metadata_fallback_used", "item_context_fallback_used",
    "item_metadata_evidence_used", "historical_review_evidence_used",
    "user_prior_evidence_used", "rating_evidence_used", "sentiment_evidence_used",
    "insufficient_review_evidence", "query_clean_is_active",
]
for column in required_false_columns:
    if queries_df[column].astype(bool).any():
        raise RuntimeError(f"Final query audit field must be false: {column}")
if not queries_df["query_evidence_source"].eq("target_review_safe_signals_only").all():
    raise RuntimeError("Final query evidence source is invalid.")
if not queries_df["query_generation_status"].eq("generated_from_review_safe_signals").all():
    raise RuntimeError("Final query generation status is invalid.")
if queries_df["query_specific_facet_family_count"].lt(MIN_SPECIFIC_FAMILIES).any():
    raise RuntimeError("Final query specific-family count is below the active review-only threshold.")
if queries_df["safe_signal_count"].lt(MIN_SPECIFIC_SIGNALS).any():
    raise RuntimeError("Final query source-signal count is below the active review-only threshold.")
if queries_df["query_generic_anchor_terms"].map(normalize_space).ne("").any():
    raise RuntimeError("Unprotected generic category anchors remain in active queries.")
if queries_df["query_generic_utility_terms"].map(normalize_space).ne("").any():
    raise RuntimeError("Unprotected generic utility terms remain in active queries.")
if queries_df["replacement_case_used"].ne(
    queries_df["replacement_source_case_id"].map(normalize_space).ne("")
).any():
    raise RuntimeError("Replacement audit fields are inconsistent.")
replacement_rows_mask = queries_df["replacement_case_used"]
if not queries_df.loc[replacement_rows_mask, "replacement_source_case_id"].eq(
    queries_df.loc[replacement_rows_mask, "case_id"]
).all():
    raise RuntimeError("replacement_source_case_id must identify the final reserve case.")
if not queries_df.loc[replacement_rows_mask, "replaced_case_id"].eq(
    queries_df.loc[replacement_rows_mask, "initial_case_id"]
).all():
    raise RuntimeError("replaced_case_id must identify the original insufficient case.")

blocking_qc_columns = [
    "brand_or_name_leak_flag", "identifier_like_leak_flag", "asin_leak_flag",
    "package_cue_flag", "seller_manufacturer_leak_flag", "exact_title_phrase_flag",
    "query_new_content_token_count", "query_rating_sentiment_term_count",
    "query_missing_multiword_phrase_count",
]
if query_qc_df[blocking_qc_columns].fillna(0).astype(int).gt(0).any().any():
    raise RuntimeError("Final leakage, no-new-content, or multiword assertion failed.")

forbidden_output_columns = {
    "target_review_text", "heldout_review_text", "review_text", "raw_review_text",
    "query_safe_text", "query_safe_facet_text", "common_functional_facet_text",
    "historical_review_reputation_text", "review_reputation_facet_text",
    "itemctx_title", "itemctx_facet_brand_text", "itemctx_identifier_diagnostic_text",
    "itemctx_manufacturer", "itemctx_store", "itemctx_package_dimensions",
    "itemctx_unit_count", "itemctx_number_of_items", "itemctx_item_model_number",
    "brand", "title", "manufacturer", "seller", "prompt", "response", "llm_response",
    "prior_review_n", "prior_history_n",
}
forbidden_output_present = sorted(forbidden_output_columns & set(queries_df.columns))
if forbidden_output_present:
    raise RuntimeError(f"Forbidden query-cache columns: {forbidden_output_present}")

comparison_rows = []
for regime in REGIME_ORDER:
    eligible_regime = ranked_candidate_df[ranked_candidate_df["regime"].eq(regime)]
    final_regime = queries_df[queries_df["regime"].eq(regime)]
    regime_qc = query_qc_df[query_qc_df["regime"].eq(regime)]
    comparison_rows.append({
        "category": "Herbal Supplements",
        "regime": regime,
        "eligible_source_cases": int(len(eligible_regime)),
        "signal_evaluated_cases": int(len(eligible_regime)),
        "unevaluated_cases": 0,
        "insufficient_cases": int((~eligible_regime["query_candidate_sufficient"]).sum()),
        "initial_selected_cases": int(eligible_regime["initial_selected"].sum()),
        "initial_insufficient_cases": int(
            (eligible_regime["initial_selected"] & ~eligible_regime["query_candidate_sufficient"]).sum()
        ),
        "replacements": int(final_regime["replacement_case_used"].sum()),
        "final_cases": int(len(final_regime)),
        "generic_anchor_rate": float(
            final_regime["query_generic_anchor_terms"].map(normalize_space).ne("").mean()
        ) if len(final_regime) else 0.0,
        "specific_cue_rate": float(
            final_regime["query_specific_facet_family_count"].gt(0).mean()
        ) if len(final_regime) else 0.0,
        "leakage_assertions_passed": bool(
            regime_qc[blocking_qc_columns].fillna(0).astype(int).eq(0).all().all()
        ),
    })
comparison_df = pd.DataFrame(comparison_rows)

query_contract = {
    "category": "Herbal Supplements",
    "evidence_scope": "target_review_safe_signals_only",
    "active_query_column": "query",
    "compatibility_query_aliases": {"query_C": "query"},
    "query_evidence_source": "target_review_safe_signals_only",
    "target_metadata_role": "post_generation_leakage_detection_and_scrubbing_only",
    "target_metadata_fallback_used": False,
    "item_context_fallback_used": False,
    "item_metadata_evidence_used": False,
    "historical_review_evidence_used": False,
    "user_prior_evidence_used": False,
    "rating_evidence_used": False,
    "sentiment_evidence_used": False,
    "raw_target_review_loaded": False,
    "raw_review_text_exported": False,
    "review_safe_seed_built_before_metadata_join": True,
    "user_prior_columns_loaded": [],
    "user_prior_columns_exported": [],
    "replacement_source_case_id_semantics": "final_same_regime_reserve_case_id; equals case_id on replacement rows",
    "replaced_case_id_semantics": "original_insufficient_selected_case_id",
    "replacement_policy": "same_regime_notebook03_locked_order",
    "initial_order_field": "initial_order_within_regime",
    "same_regime_replacement_order_field": "same_regime_replacement_order",
    "selection_stage_source": "notebook03_deferred_to_query_audit",
    "generic_anchor_rescue_enabled": False,
    "item_context_seed_enabled": False,
    "flavor_family_used_for_query": False,
    "excluded_query_families": EXCLUDED_QUERY_FAMILIES,
    "multiword_entity_preservation_required": True,
    "query_clean_column": "query_clean",
    "query_clean_is_active": False,
    "final_min_query_tokens": MIN_QUERY_TOKENS,
    "final_max_query_tokens": MAX_QUERY_TOKENS,
    "final_min_query_safe_signal_families": MIN_SPECIFIC_FAMILIES,
    "final_min_query_safe_signal_total": MIN_SPECIFIC_SIGNALS,
    "regime_order": REGIME_ORDER,
    "eligible_regime_counts": EXPECTED_ELIGIBLE_REGIME_COUNTS,
    "target_per_regime": EXPECTED_TARGET_PER_REGIME,
    "query_variant": QUERY_VARIANT,
    "eligibility_policy": ELIGIBILITY_POLICY,
    "post_metadata_scrub_residual_repair_enabled": True,
    "residual_repair_source": RESIDUAL_TEXT_COLUMN,
    "active_feasible_counts_before_generation": {
        regime: int(active_feasible_counts.loc[regime]) for regime in REGIME_ORDER
    },
    "selection_feasible_counts_before_generation": {
        regime: int(selection_feasible_counts.loc[regime]) for regime in REGIME_ORDER
    },
    "downstream_prior_item_guard": "non-cold final cases require at least one Notebook03 training-safe prior item present in Notebook04 item_docs_herbal",
    "dspy_mode": DSPY_MODE,
    "dspy_model": DEEPSEEK_MODEL,
    "dspy_temperature": DSPY_TEMPERATURE,
    "dspy_max_tokens": DSPY_MAX_TOKENS,
    "dspy_max_retries": DSPY_MAX_RETRIES,
    "runtime_comparison": comparison_df.to_dict(orient="records"),
    "output_paths": {
        "query_cache": str(QUERY_CACHE_PATH),
        "query_cache_csv": str(QUERY_CACHE_CSV_PATH),
        "query_qc": str(QUERY_QC_PATH),
        "contract": str(QUERY_CONTRACT_PATH),
        "summary": str(QUERY_SUMMARY_PATH),
        "insufficient_evidence_audit": str(OUTPUT_DIR / "herbal_query_insufficient_review_evidence.csv"),
        "replacement_mapping": str(OUTPUT_DIR / "herbal_query_replacement_mapping.csv"),
        "comparison": str(OUTPUT_DIR / "herbal_query_generation_comparison.csv"),
        "pre_generation_failure_reason_summary": str(PRE_GENERATION_FAILURE_REASON_PATH),
        "pre_generation_attrition_qc": str(PRE_GENERATION_ATTRITION_QC_PATH),
    },
}

query_summary = {
    "category": "Herbal Supplements",
    "eligible_source_rows": int(len(ranked_candidate_df)),
    "eligibility_policy": ELIGIBILITY_POLICY,
    "pre_generation_attrition_qc_path": str(PRE_GENERATION_ATTRITION_QC_PATH),
    "active_feasible_counts_before_generation": {
        regime: int(active_feasible_counts.loc[regime]) for regime in REGIME_ORDER
    },
    "selection_feasible_counts_before_generation": {
        regime: int(selection_feasible_counts.loc[regime]) for regime in REGIME_ORDER
    },
    "downstream_prior_item_guard": "non-cold final cases require at least one Notebook03 training-safe prior item present in Notebook04 item_docs_herbal",
    "insufficient_rows": int((~ranked_candidate_df["query_candidate_sufficient"]).sum()),
    "replacement_rows": int(queries_df["replacement_case_used"].sum()),
    "final_rows": int(len(queries_df)),
    "regime_comparison": comparison_df.to_dict(orient="records"),
}

print("Validation: PASS")


In [ ]:
# ==== Gate and Export Canonical Query Artifacts ====
if not IDENTITY_PREFLIGHT_PASSED:
    raise RuntimeError("Canonical query publication is blocked until SC-0 identity preflight passes.")
should_write_outputs = (
    bool(IDENTITY_PREFLIGHT_PASSED)
    and bool(ALLOW_CANONICAL_PUBLICATION)
    and not bool(RUN_IDENTITY_ONLY_PREFLIGHT)
    and ((not RUN_SMOKE_TEST) or WRITE_OUTPUTS_IN_SMOKE_TEST)
)
if should_write_outputs:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    queries_df.to_parquet(QUERY_CACHE_PATH, index=False)
    queries_df.to_csv(QUERY_CACHE_CSV_PATH, index=False, encoding="utf-8-sig")
    query_qc_df.to_parquet(QUERY_QC_PATH, index=False)
    query_qc_df.to_csv(
        OUTPUT_DIR / "herbal_query_generation_qc_medium_heavy_dspy.csv",
        index=False,
        encoding="utf-8-sig",
    )
    comparison_df.to_csv(
        OUTPUT_DIR / "herbal_query_generation_comparison.csv",
        index=False,
        encoding="utf-8-sig",
    )
    queries_df.head(100).to_csv(
        OUTPUT_DIR / "herbal_query_generation_preview_medium_heavy_dspy.csv",
        index=False,
        encoding="utf-8-sig",
    )
    queries_df[[
        "case_id", "query", "query_C", "query_clean", "query_clean_is_active",
        "query_generic_anchor_terms", "query_generic_utility_terms",
    ]].to_csv(
        OUTPUT_DIR / "herbal_query_C_clean_audit.csv",
        index=False,
        encoding="utf-8-sig",
    )
    comparison_df.to_csv(
        OUTPUT_DIR / "herbal_query_generation_guardrail_summary_medium_heavy_dspy.csv",
        index=False,
        encoding="utf-8-sig",
    )
    with open(QUERY_CONTRACT_PATH, "w", encoding="utf-8") as file:
        json.dump(query_contract, file, ensure_ascii=False, indent=2)
    with open(QUERY_SUMMARY_PATH, "w", encoding="utf-8") as file:
        json.dump(query_summary, file, ensure_ascii=False, indent=2, default=str)

    expected_outputs = [
        QUERY_CACHE_PATH,
        QUERY_CACHE_CSV_PATH,
        QUERY_QC_PATH,
        QUERY_CONTRACT_PATH,
        QUERY_SUMMARY_PATH,
        PRE_GENERATION_FAILURE_REASON_PATH,
        PRE_GENERATION_ATTRITION_QC_PATH,
    ]
    missing_outputs = [str(path) for path in expected_outputs if not path.exists()]
    if missing_outputs:
        raise RuntimeError(f"Missing query-generation outputs: {missing_outputs}")

if should_write_outputs:
    print("Output:", QUERY_CACHE_PATH)
    print("Output:", QUERY_QC_PATH)
    print("Output:", PRE_GENERATION_FAILURE_REASON_PATH)
else:
    print("SC-0 identity preflight mode: canonical query outputs were not written.")
print("Rows:", len(queries_df))
